### Setup


In [2]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [3]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data

### Load and prepare data

In [4]:
sim_results_folder = '../results/simulation'
data_folder = '../data'
sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [10]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()
data = load_data(emp_at_file)

# Filter data for learning phase (Sessions 1 to 8)
data_learning = data[data['SessionID'] <= 8].copy()

# Map synchrony values to each condition in DataFrame
data_learning['Synchrony'] = data_learning['Condition'].apply(lambda x: sync_results_vector[x-1])

# Z-score the relevant columns
data_learning = zscore_data(data_learning, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony'])

# Center the session number
data_learning['SessionID'] = data_learning['SessionID'] - data_learning['SessionID'].mean()

# Dummy code session 9
data['Transfer'] = (data['SessionID'] == 9)

### Define statistical models

In [11]:
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + SessionID * (ContrastHeterogeneity + GridCoarseness) +  (1 + SessionID + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

Does learning occur (i.e., does session have an effect on performance)? 

Do the effects of contrast heterogeneity and/or grid coarseness depend on session?

In [ ]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.9,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, SessionID, SessionID:ContrastHeterogeneity, SessionID:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, SessionID|SubjectID_sigma, SessionID|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]


In [ ]:
predictors = ["SessionID", "ContrastHeterogeneity", "GridCoarseness", "SessionID:ContrastHeterogeneity","SessionID:GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['greater', 'less', 'less','less','greater', 'greater']

posterior = posterior_table(idata_features, predictors, directions)

odds_ratios = OR_table(idata_features, predictors)

print("One-sided posterior probabilities:")
print(posterior)
print("\nOdds ratios:")
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)